# REDACT Library — Full Pipeline Demo

End-to-end dataset generation using the high-level pipeline functions.

This notebook demonstrates:
1. **Content moderation inputs** — generate harmful prompts across taxonomy categories
2. **Output responses** — generate model responses for the input samples
3. **Jailbreaks** — apply obfuscation, hacking, and manipulation techniques
4. **Complete dataset** — merge everything into a single CSV

Requires `VENICE_API_KEY` set in environment or `.env` file.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from Redact_Library import set_seed
set_seed(42)

## Configuration

Adjust these settings before running. Small values are set for demo purposes.

In [ ]:
# --- Pipeline settings ---
MODEL = "venice-uncensored"          # Generation model
BASE_URL = "https://api.venice.ai/api/v1"

# Content moderation input
SAMPLES_PER_CATEGORY = 15            # Target accepted samples per category
NUM_CATEGORIES = 3                   # Number of categories to process (None = all)
USE_METAPROMPT = True                # True = LLM generates descriptions + seeds
NUM_SEEDS = 8                        # Number of seed prompts to generate (metaprompt mode)
SAMPLES_PER_REQUEST = 5              # Samples requested per LLM call
FRESH_RUN = True                     # Clear existing data before generating (avoids inflated rejections)

# Output responses
MAX_OUTPUT_SAMPLES = 10              # Limit output generation (None = all)

# Jailbreaks
TECHNIQUE_TYPES = ["obfuscation", "hacking"]  # Which families to run
MAX_PER_TECHNIQUE = None             # Limit per individual technique (None = all)

## Step 1: Generate Content Moderation Inputs

Uses the default `content_moderation_categories` taxonomy. Each category goes through:
1. LLM-generated category description (metaprompt mode)
2. LLM-generated seed prompts (`NUM_SEEDS` controls how many)
3. Sample generation with quality checking

The pipeline keeps generating turns until `SAMPLES_PER_CATEGORY` accepted samples
are reached. Each turn requests `SAMPLES_PER_REQUEST` samples, but some may be
rejected by the quality checker (duplicates, off-topic, low quality). The pipeline
automatically retries with feedback from rejected samples to improve the next turn.

Results are saved per-category to `Redact_Library/Datasets/{category}/samples.csv`.

In [ ]:
from Redact_Library import generate_inputs

inputs = generate_inputs(
    samples_per_category=SAMPLES_PER_CATEGORY,
    num_categories=NUM_CATEGORIES,
    use_metaprompt=USE_METAPROMPT,
    num_seeds=NUM_SEEDS,
    samples_per_request=SAMPLES_PER_REQUEST,
    model=MODEL,
    base_url=BASE_URL,
    fresh=FRESH_RUN,
)

print(f"\nGenerated {len(inputs)} accepted input samples")
inputs.head(10)

## Step 2: Generate Output Responses

Runs the model on each input sample to generate a response.
Saved to `Redact_Library/Datasets/output_responses.csv`.

In [ ]:
from Redact_Library import generate_outputs

outputs = generate_outputs(
    inputs=inputs,
    model=MODEL,
    base_url=BASE_URL,
    max_samples=MAX_OUTPUT_SAMPLES,
)

print(f"\nGenerated {len(outputs)} output responses")
outputs.head(5)

## Step 3: Generate Jailbreaks

Applies jailbreak techniques to the input prompts. Techniques include:
- **Obfuscation**: encoding, translation (low-resource languages), structural wrapping, ascii art, tokenbreak, adversarial suffixes
- **Hacking**: persona roleplay, hypothetical framing, authority obedience, AVI, deep inception
- **Manipulation**: FSH (few-shot hacking), DAP (distract and persuade)

Saved per-technique to `Redact_Library/Datasets/jailbreaks/{technique}.csv`.

In [ ]:
from Redact_Library import generate_jailbreaks

jailbreaks = generate_jailbreaks(
    inputs=inputs,
    technique_types=TECHNIQUE_TYPES,
    max_samples_per_technique=MAX_PER_TECHNIQUE,
    model=MODEL,
    base_url=BASE_URL,
)

print(f"\nGenerated {len(jailbreaks)} jailbreak samples")
jailbreaks.head(10)

## Step 4: Build Complete Dataset

Merges all generated data (inputs, outputs, jailbreaks) from their saved CSVs into a single dataset.

In [ ]:
from Redact_Library import build_dataset

dataset = build_dataset()

print(f"\nComplete dataset: {len(dataset)} samples")
dataset.head(20)

## Dataset Statistics

In [ ]:
if not dataset.empty:
    print("=== By Dataset Type ===")
    if "dataset_type" in dataset.columns:
        print(dataset["dataset_type"].value_counts().to_string())

    print("\n=== By Category ===")
    if "category" in dataset.columns:
        print(dataset["category"].value_counts().to_string())

    print("\n=== By Technique (jailbreaks only) ===")
    if "technique" in dataset.columns:
        jb = dataset[dataset["dataset_type"] == "jailbreak"] if "dataset_type" in dataset.columns else dataset
        print(jb["technique"].value_counts().to_string())
else:
    print("No data generated yet.")

---

# Custom Taxonomy

You can define your own taxonomy and run the same pipeline on it. This is useful for generating datasets for specific domains beyond content moderation.

In [ ]:
from Redact_Library import create_taxonomy

taxonomy = create_taxonomy(
    name="financial_harm",
    description="Prompts related to financial fraud and exploitation.",
    categories={
        "Investment Scams": {
            "description": "Prompts about fraudulent investment schemes, Ponzi schemes, pump-and-dump, fake ICOs.",
            "subcategories": ["Ponzi Schemes", "Pump and Dump", "Fake Crypto"],
        },
        "Identity Theft": {
            "description": "Prompts about stealing personal information for financial gain, phishing, social engineering.",
            "subcategories": ["Phishing", "Social Engineering", "Document Forgery"],
        },
    },
)

print(f"\nTaxonomy created with {len(taxonomy['categories'])} categories")

In [ ]:
custom_inputs = generate_inputs(
    taxonomy=taxonomy,
    samples_per_category=5,
    num_categories=2,
    model=MODEL,
    base_url=BASE_URL,
    fresh=True,
)

print(f"\nGenerated {len(custom_inputs)} samples for custom taxonomy")
custom_inputs.head()